# Training temporal (GRU) motion models

Per-frame MLPs (the `build_baseline` checkpoints) see no temporal context, so they flicker. This notebook trains a GRU that regresses the **center-frame** quaternions from a window `±w` of scaled feature frames, then compares rotation-angle error against the baseline on the same animation-level split.

Run top-to-bottom. Set `CFG` values in the config cell, then execute.

## 0. Setup

In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

# walk up from cwd until we find the project root (contains pyproject.toml)
ROOT = Path.cwd().resolve()
for ancestor in [ROOT, *ROOT.parents]:
    if (ancestor / "pyproject.toml").exists():
        ROOT = ancestor
        break
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("root:", ROOT)
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())

root: A:\projects\ExtendedMocap\notebooks
torch: 2.13.0+cu132 | cuda: True


## 1. Config

In [2]:
from extended_mocap.training import TrainConfig, SEGMENTS, train_segment, load_data, segment_bones, evaluate_segment_report, save_checkpoint, write_metrics
from extended_mocap.evaluation import segment_subset, _quat_block_view

CFG = TrainConfig(
    window=6,
    hidden=128,
    layers=2,
    epochs=30,
    stride=3,
    batch_size=128,
    lr=1e-3,
    test_size=0.2,
    seed=42,
    device="auto",
)

DATA_MP = ROOT / "data" / "mediapipe" / "csv"
DATA_MOCAP = ROOT / "data" / "mocap" / "csv"
CACHE = ROOT / "data" / "cache"
OUT_DIR = ROOT / "models" / "temporal"
BASE_METRICS = ROOT / "models" / "baseline" / "metrics.json"

print(CFG)

TrainConfig(window=6, hidden=128, layers=2, epochs=30, stride=3, batch_size=128, lr=0.001, test_size=0.2, seed=42, device='auto', verbose=True)


## 2. Load data + scaler

In [3]:
samples, feature_columns, train, test, scaler = load_data(
    str(DATA_MP), str(DATA_MOCAP), str(CACHE), CFG
)
print(f"animations: {len(samples)} | motions: {len({s.motion_id for s in samples})} | train {len(train)} ({len({s.motion_id for s in train})} motions) | test {len(test)} ({len({s.motion_id for s in test})} motions)")
print(f"features: {len(feature_columns)}")
print(f"bones: {len(samples[0].bones)}")

FileNotFoundError: [WinError 3] Системе не удается найти указанный путь: 'A:\\projects\\ExtendedMocap\\notebooks\\data\\mediapipe\\csv'

## 3. Train one segment (with per-step loss captured)

Re-run this cell per segment, or loop over all three below.

In [ ]:
SEGMENT = "body"  # one of body / left_hand / right_hand
bones = segment_bones(samples, SEGMENT)
subsets = [segment_subset(scaler.transform(s.features), feature_columns, SEGMENT) for s in train]
targets = [_quat_block_view(s, bones) for s in train]
print(f"{SEGMENT}: in={subsets[0].shape[1]} out={targets[0].shape[1]} n_bones={len(bones)}")

model, losses = train_segment(subsets, targets, CFG)
print(f"training done: {len(losses)} optimizer steps")

### Loss curve

In [ ]:
def epoch_means(losses, n_epochs):
    steps = max(1, len(losses) // n_epochs)
    return [float(np.mean(losses[i*steps:(i+1)*steps])) for i in range(n_epochs)]

em = epoch_means(losses, CFG.epochs)
plt.figure(figsize=(9,4))
plt.plot(range(1, CFG.epochs+1), em, marker='o')
plt.xlabel('epoch'); plt.ylabel('MSE (train)'); plt.title(f'{SEGMENT} loss per epoch')
plt.grid(alpha=.3); plt.show()

### Evaluate on held-out test animations

In [ ]:
scoring = evaluate_segment_report(model, test, feature_columns, scaler, SEGMENT, bones, CFG.window, reduce="motion")
print(f"mean {scoring['mean_deg']:.2f} deg | median {scoring['median_deg']:.2f} | p90 {scoring['p90_deg']:.2f}")
print(f"worst bones:")
for err, b in sorted((v['mean_deg'], b) for b, v in scoring['bones'].items())[-5:]:
    print(f"  {b:28s} {err:6.2f} deg")

## 4. Train all segments + save checkpoints + write metrics

In [ ]:
report = {}
history = {}
for segment in SEGMENTS:
    bones = segment_bones(samples, segment)
    subsets = [segment_subset(scaler.transform(s.features), feature_columns, segment) for s in train]
    targets = [_quat_block_view(s, bones) for s in train]
    model, losses = train_segment(subsets, targets, CFG)
    history[segment] = epoch_means(losses, CFG.epochs)
    save_checkpoint(OUT_DIR, segment, model, bones, feature_columns, scaler, CFG)

    scoring = evaluate_segment_report(model, test, feature_columns, scaler, segment, bones, CFG.window, reduce="motion")
    report[segment] = {k: v for k, v in scoring.items() if k != 'bones'}
    report[segment]['bones'] = scoring['bones']
    print(f"[{segment}] mean={scoring['mean_deg']:.2f} deg | median={scoring['median_deg']:.2f} | p90={scoring['p90_deg']:.2f}")

write_metrics(OUT_DIR, report)

### All-segment loss + baseline comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for segment in SEGMENTS:
    axes[0].plot(range(1, CFG.epochs+1), history[segment], marker='.', label=segment)
axes[0].set_xlabel('epoch'); axes[0].set_ylabel('MSE'); axes[0].set_title('train loss / epoch')
axes[0].grid(alpha=.3); axes[0].legend()

baseline = json.loads(BASE_METRICS.read_text()) if BASE_METRICS.exists() else None
if baseline is not None:
    labels = SEGMENTS
    t = [report[s]['mean_deg'] for s in SEGMENTS]
    b = [baseline[s]['mean_deg'] for s in SEGMENTS]
    x = np.arange(len(SEGMENTS)); w = .35
    axes[1].bar(x - w/2, b, w, label='baseline', color='#999')
    axes[1].bar(x + w/2, t, w, label='temporal', color='#4682b4')
    axes[1].set_xticks(x); axes[1].set_xticklabels(SEGMENTS)
    axes[1].set_ylabel('mean rotation error (deg)')
    axes[1].set_title('test mean error vs baseline')
    for xi, (tb, tt) in enumerate(zip(b, t)):
        axes[1].text(xi - w/2, tb, f'{tb:.1f}', ha='center', va='bottom', fontsize=8)
        axes[1].text(xi + w/2, tt, f'{tt:.1f}', ha='center', va='bottom', fontsize=8)
    axes[1].legend()
else:
    axes[1].text(.5, .5, 'baseline metrics.json missing', ha='center')
    axes[1].axis('off')
plt.tight_layout(); plt.show()

## 5. Per-bone error table

In [ ]:
rows = []
for segment, rep in report.items():
    for bone, v in rep['bones'].items():
        rows.append({'segment': segment, 'bone': bone, 'mean_deg': v['mean_deg'], 'median_deg': v['median_deg']})
df = pd.DataFrame(rows).sort_values('mean_deg', ascending=False)
df.head(20)

## 6. Reuse in the production pipeline

`MocapInferer` auto-detects this checkpoint format (dict with `arch` + `state_dict`, plus `feature_columns.json` / `scaler.npz` / `<segment>_bones.json` written by `save_checkpoint`).

In [ ]:
from extended_mocap.inference import MocapInferer
infer = MocapInferer(model_config={
    s: {"path": str(OUT_DIR / f"{s}.pt")} for s in SEGMENTS
}, model_dir=str(OUT_DIR))
print("MocapInferer built with temporal checkpoints:", list(infer.segments))